In [2]:
!pip install -q llama-cpp-python sentence-transformers faiss-gpu-cu12 rank-bm25 pandas numpy fastapi uvicorn pyngrok psutil cloudscraper beautifulsoup4 lxml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 40.4 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 40.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.5 MB/s eta 0:00:00


In [3]:
import urllib.parse

def create_wb_link(product_name: str) -> str:
    """Создает ссылку на Wildberries с правильным кодированием"""
    base_url = "https://www.wildberries.ru/catalog/0/search.aspx?search="
    # Очищаем название от лишних символов
    clean_name = product_name.split(',')[0].strip().strip('"')
    # Кодируем для URL
    encoded_name = urllib.parse.quote(clean_name)
    return base_url + encoded_name

def create_yamarket_link(product_name: str) -> str:
    """Создает ссылку на Яндекс Маркет с правильным кодированием"""
    base_url = "https://market.yandex.ru/search?text="
    # Очищаем название от лишних символов
    clean_name = product_name.split(',')[0].strip().strip('"')
    # Для Яндекс Маркета можно использовать + или кодирование
    encoded_name = urllib.parse.quote(clean_name)
    return base_url + encoded_name

def create_ozon_link(product_name: str) -> str:
    """Создает ссылку на Ozon с правильным кодированием"""
    base_url = "https://www.ozon.ru/search/?text="
    # Очищаем название от лишних символов
    clean_name = product_name.split(',')[0].strip().strip('"')
    # Кодируем для URL
    encoded_name = urllib.parse.quote(clean_name)
    return base_url + encoded_name

def create_all_links(product_name: str) -> dict:
    """Создает ссылки на все маркетплейсы"""
    return {
        "wildberries": create_wb_link(product_name),
        "yamarket": create_yamarket_link(product_name),
        "ozon": create_ozon_link(product_name)
    }

# Пример использования:
product = "Подарочный набор для женского ухода"
links = create_all_links(product)
print(f"Wildberries: {links['wildberries']}")
print(f"Яндекс Маркет: {links['yamarket']}")
print(f"Ozon: {links['ozon']}")

Wildberries: https://www.wildberries.ru/catalog/0/search.aspx?search=%D0%9F%D0%BE%D0%B4%D0%B0%D1%80%D0%BE%D1%87%D0%BD%D1%8B%D0%B9%20%D0%BD%D0%B0%D0%B1%D0%BE%D1%80%20%D0%B4%D0%BB%D1%8F%20%D0%B6%D0%B5%D0%BD%D1%81%D0%BA%D0%BE%D0%B3%D0%BE%20%D1%83%D1%85%D0%BE%D0%B4%D0%B0
Яндекс Маркет: https://market.yandex.ru/search?text=%D0%9F%D0%BE%D0%B4%D0%B0%D1%80%D0%BE%D1%87%D0%BD%D1%8B%D0%B9%20%D0%BD%D0%B0%D0%B1%D0%BE%D1%80%20%D0%B4%D0%BB%D1%8F%20%D0%B6%D0%B5%D0%BD%D1%81%D0%BA%D0%BE%D0%B3%D0%BE%20%D1%83%D1%85%D0%BE%D0%B4%D0%B0
Ozon: https://www.ozon.ru/search/?text=%D0%9F%D0%BE%D0%B4%D0%B0%D1%80%D0%BE%D1%87%D0%BD%D1%8B%D0%B9%20%D0%BD%D0%B0%D0%B1%D0%BE%D1%80%20%D0%B4%D0%BB%D1%8F%20%D0%B6%D0%B5%D0%BD%D1%81%D0%BA%D0%BE%D0%B3%D0%BE%20%D1%83%D1%85%D0%BE%D0%B4%D0%B0


In [4]:
def create_search_links_for_idea(idea_title: str) -> str:
    """Создает Markdown-строку со ссылками на поиск идеи на маркетплейсах"""
    # Очищаем название от лишних символов (** и кавычек)
    clean_title = idea_title.strip().strip('*').strip('"')
    
    # Создаем ссылки для каждого маркетплейса
    wb_link = f"https://www.wildberries.ru/catalog/0/search.aspx?search={urllib.parse.quote(clean_title)}"
    ym_link = f"https://market.yandex.ru/search?text={urllib.parse.quote(clean_title).replace('%20', '+')}"
    ozon_link = f"https://www.ozon.ru/search/?text={urllib.parse.quote(clean_title)}"
    
    # Формируем Markdown-строку с правильным заголовком
    return (f"\n\n🔗 **Найти этот подарок:**\n"
            f"  • [Wildberries]({wb_link}) | "
            f"[Яндекс Маркет]({ym_link}) | "
            f"[Ozon]({ozon_link})")

In [5]:
import urllib.parse

def create_market_links(product_name: str) -> dict:
    """Универсальная функция для создания ссылок на все маркетплейсы"""
    # Очищаем название
    clean_name = product_name.split(',')[0].strip().strip('"')
    
    # Кодируем для URL
    encoded_wb = urllib.parse.quote(clean_name)
    encoded_ya = urllib.parse.quote(clean_name).replace('%20', '+')  # Яндекс любит +
    encoded_ozon = urllib.parse.quote(clean_name)
    
    return {
        "wildberries": f"https://www.wildberries.ru/catalog/0/search.aspx?search={encoded_wb}",
        "yamarket": f"https://market.yandex.ru/search?text={encoded_ya}",
        "ozon": f"https://www.ozon.ru/search/?text={encoded_ozon}"
    }

# Пример
product = "Кофемашина DeLonghi"
links = create_market_links(product)
print(f"WB: {links['wildberries']}")
print(f"YM: {links['yamarket']}")
print(f"OZ: {links['ozon']}")

WB: https://www.wildberries.ru/catalog/0/search.aspx?search=%D0%9A%D0%BE%D1%84%D0%B5%D0%BC%D0%B0%D1%88%D0%B8%D0%BD%D0%B0%20DeLonghi
YM: https://market.yandex.ru/search?text=%D0%9A%D0%BE%D1%84%D0%B5%D0%BC%D0%B0%D1%88%D0%B8%D0%BD%D0%B0+DeLonghi
OZ: https://www.ozon.ru/search/?text=%D0%9A%D0%BE%D1%84%D0%B5%D0%BC%D0%B0%D1%88%D0%B8%D0%BD%D0%B0%20DeLonghi


In [17]:
%%writefile rag_pipeline.py

import torch
import pandas as pd
import numpy as np
import json
import re
import faiss
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import gc
import os
import psutil
from openai import OpenAI
import time
import requests
import urllib.parse
import random
import cloudscraper
from bs4 import BeautifulSoup

# 3. Настройки модели
MODEL_NAME = "llama-3.3-70b-versatile"
EMBED_NAME = "ai-forever/ru-en-RoSBERTa"
BASE_URL = "https://api.groq.com/openai/v1"

# ========== НАСТРОЙКИ NGROK ==========
# Получить токен на https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "3AXU3ZzWePTsFJy3a2DMs73OgFC_74JVHeCDctdgesbSLJLdD"

# ========== НАСТРОЙКИ (ИЗМЕНИТЕ ПОД СЕБЯ) ==========
# 1. Groq API ключ - получить на https://console.groq.com
GROQ_API_KEY = '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'

# 2. Путь к датасету в Kaggle
DATA_PATH = "/kaggle/input/datasets/ursofiia/gift-genius-dataset/gifts_dataset_ru_plus (1).csv"

# 4. Параметры генерации
GEN_CONFIG = dict(temperature=0.9, top_p=0.95)
FINAL_TOP_K = 15

# ========== КОНФИГУРАЦИЯ УСТРОЙСТВА ==========
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_GPU = DEVICE == "cuda"
print(f"Устройство: {DEVICE}")

# ========== ИНИЦИАЛИЗАЦИЯ GROQ CLIENT ==========
groq_client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url=BASE_URL,
    max_retries=3
)

# ========== ИНИЦИАЛИЗАЦИЯ SCRAPER ==========
scraper = cloudscraper.create_scraper()

# ========== ФУНКЦИИ ДЛЯ НОРМАЛИЗАЦИИ НАЗВАНИЙ ==========
def normalize_product_name(name):
    """Очищает название товара от лишних символов"""
    # Заменяем разные виды дефисов/тире на обычный дефис
    name = re.sub(r'[‑–—]', '-', name)
    # Убираем лишние пробелы
    name = re.sub(r'\s+', ' ', name).strip()
    return name

# ========== ФУНКЦИИ ДЛЯ ССЫЛОК ==========
def create_wb_link(search_query: str) -> str:
    """Создает ссылку на Wildberries для поискового запроса"""
    clean_query = normalize_product_name(search_query.split(',')[0].strip().strip('"'))
    encoded_query = urllib.parse.quote(clean_query)
    return "https://www.wildberries.ru/catalog/0/search.aspx?search=" + encoded_query

def create_yamarket_link(search_query: str) -> str:
    """Создает ссылку на Яндекс Маркет для поискового запроса"""
    clean_query = normalize_product_name(search_query.split(',')[0].strip().strip('"'))
    encoded_query = urllib.parse.quote(clean_query).replace('%20', '+')
    return "https://market.yandex.ru/search?text=" + encoded_query

def create_ozon_link(search_query: str) -> str:
    """Создает ссылку на Ozon для поискового запроса"""
    clean_query = normalize_product_name(search_query.split(',')[0].strip().strip('"'))
    encoded_query = urllib.parse.quote(clean_query)
    return "https://www.ozon.ru/search/?text=" + encoded_query

def create_all_links(search_query: str) -> dict:
    """Создает ссылки на все маркетплейсы для поискового запроса"""
    clean_query = normalize_product_name(search_query.split(',')[0].strip().strip('"'))
    return {
        "wildberries": create_wb_link(clean_query),
        "yamarket": create_yamarket_link(clean_query),
        "ozon": create_ozon_link(clean_query)
    }

# ========== ФУНКЦИИ ДЛЯ ПАРСИНГА ЦЕН ==========
def fetch_yandex_price(product_name):
    """Получает цену товара с Яндекс Маркета"""
    url = create_yamarket_link(product_name)
    try:
        # Задержка чтобы не заблокировали
        time.sleep(random.uniform(2, 4))
        response = scraper.get(url, timeout=15)
        
        if response.status_code != 200:
            print(f"  ⚠️ Статус {response.status_code} для {product_name[:30]}...")
            return None

        soup = BeautifulSoup(response.text, 'lxml')
        
        # Основные селекторы для поиска цены
        selectors = [
            'span.ds-text_color_price-term',
            'span.ds-text_weight_bold',
            '[data-autotest-id="price"]',
            'span.ds-text_price-term',
            'span[data-baobab-name="price"]',
            'h3 span[data-autotest-id="price"]',
            'div[data-zone-name="price"] span',
            'span[data-zone-name="price"]'
        ]
        
        price_element = None
        for selector in selectors:
            price_element = soup.select_one(selector)
            if price_element:
                break
        
        if price_element:
            price_text = price_element.get_text(strip=True)
            # Ищем цифры в тексте
            digits = re.findall(r'(\d[\d\s]*)\s*[₽руб]', price_text)
            if digits:
                # Убираем пробелы из цифр
                clean_price = re.sub(r'\s', '', digits[0])
                return f"{clean_price} ₽"
        
        # JSON-LD fallback
        script = soup.find('script', type='application/ld+json')
        if script and script.string:
            try:
                data = json.loads(script.string)
                if 'offers' in data:
                    price = data['offers'].get('price')
                    if price:
                        return f"{price} ₽"
                if isinstance(data.get('offers'), list) and data['offers']:
                    price = data['offers'][0].get('price')
                    if price:
                        return f"{price} ₽"
            except:
                pass
        
        return None
    except Exception as e:
        print(f"  ⚠️ Ошибка парсинга: {e}")
        return None

def fetch_wb_price(product_name):
    """Получает цену товара с Wildberries (заглушка - требует API)"""
    # Wildberries требует API или более сложный парсинг
    return None

def fetch_ozon_price(product_name):
    """Получает цену товара с Ozon (заглушка - требует API)"""
    # Ozon требует API или более сложный парсинг
    return None

def get_price(product_name, marketplace="yandex"):
    """Получает цену товара с указанного маркетплейса"""
    if marketplace == "yandex":
        return fetch_yandex_price(product_name)
    elif marketplace == "wildberries":
        return fetch_wb_price(product_name)
    elif marketplace == "ozon":
        return fetch_ozon_price(product_name)
    return None

# ========== ЗАГРУЗКА ДАННЫХ ==========
print("\n" + "=" * 60)
print("📊 ЗАГРУЗКА ДАННЫХ")
print("=" * 60)
df = pd.read_csv(DATA_PATH).fillna("")
print(f"✅ Подарков в базе: {len(df)}")
df["_doc"] = df.astype(str).agg(" | ".join, axis=1).str.strip()
corpus = df["_doc"].tolist()

# ========== EMBEDDING МОДЕЛЬ ==========
print("\n" + "=" * 60)
print("🔤 ЗАГРУЗКА EMBEDDING МОДЕЛИ")
print("=" * 60)
embed_model = SentenceTransformer(EMBED_NAME, device=DEVICE)
print(f"✅ Модель загружена")

def encode(texts, instruction=None, batch_size=16):
    if instruction:
        texts = [f"{instruction}\n{t}" for t in texts]
    return embed_model.encode(texts, batch_size=batch_size, normalize_embeddings=True)

QUERY_INSTR = "Retrieve gift ideas that best match the user request"

# ========== СОЗДАНИЕ ИНДЕКСОВ ==========
print("\n" + "=" * 60)
print("🔍 СОЗДАНИЕ ИНДЕКСОВ")
print("=" * 60)
doc_embs = encode(corpus, batch_size=32)
print(f"✅ Эмбеддинги: {doc_embs.shape}")

faiss_index = faiss.IndexFlatIP(doc_embs.shape[1])
faiss_index.add(doc_embs)
bm25 = BM25Okapi([t.lower().split() for t in corpus])
print(f"✅ Индексы готовы")

# ========== RETRIEVAL ==========
def retrieve_for_query(query_text):
    q_emb = encode([query_text], instruction=QUERY_INSTR)
    sims, idxs = faiss_index.search(q_emb, FINAL_TOP_K * 2)
    return idxs[0].tolist()

def get_products_context(idx_list, top_k=FINAL_TOP_K):
    context_df = df.iloc[idx_list[:top_k]].copy()
    products = []
    for _, row in context_df.iterrows():
        name = str(row.iloc[0]).split(',')[0].strip().strip('"')
        products.append(name)
    return products, context_df

# ========== EXTRACTORS ==========
def extract_budget(query):
    patterns = [
        r"(?:до|не более|максимум)\s+(\d[\d\s]*)\s*(?:руб|р\b|₽|тыс|k|к)",
        r"(\d[\d\s]*)\s*(?:–|-)\s*(\d[\d\s]*)\s*(?:руб|р\b|₽)",
        r"бюджет[: ]+(\d[\d\s]*)\s*(?:руб|р\b|₽|тыс|k|к)?",
    ]
    for p in patterns:
        m = re.search(p, query, re.IGNORECASE)
        if m:
            return m.group(0).strip()
    return None

OCCASION_MAP = {
    "день рождения": "День рождения", "новый год": "Новый год", "свадьба": "Свадьба",
    "8 марта": "8 марта", "23 февраля": "23 февраля", "юбилей": "Юбилей",
    "годовщина": "Годовщина", "рождение ребенка": "Рождение ребенка"
}

def detect_occasion(query):
    q = query.lower()
    for kw, label in OCCASION_MAP.items():
        if kw in q:
            return label
    return None

_RECIPIENT_HINTS = ["маме", "папе", "другу", "подруге", "коллеге", "брату", "сестре", 
                    "мужу", "жене", "дедушке", "бабушке", "ребёнку", "девушке", "парню"]

def is_query_too_vague(query):
    words = query.lower().split()
    has_who = any(h in words for h in _RECIPIENT_HINTS)
    return len(words) < 4 and not has_who

# ========== GROQ CALL ==========
def call_groq(system_prompt, user_prompt, max_tokens=1000):
    try:
        response = groq_client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=GEN_CONFIG["temperature"],
            top_p=GEN_CONFIG["top_p"],
            max_tokens=max_tokens,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"⚠️ Ошибка Groq: {e}")
        return ""

# ========== PERSONA PROMPT ==========
PERSONA_SYSTEM = """Ты -- психолог. Создай детальный портрет человека для подбора подарка.
Опиши:
1. Возраст и пол
2. Характер и образ жизни
3. Интересы и увлечения
4. Скрытые желания и мечты
5. Что он/она точно НЕ любит (если указано в запросе)

Портрет должен быть подробным (3-5 предложений)."""

def persona_prompt(query, occasion=None, budget=None):
    ctx = ""
    if occasion:
        ctx += f"Повод: {occasion}\n"
    if budget:
        ctx += f"Бюджет: {budget}\n"
    return f'Запрос: "{query}"\n\n{ctx}Опиши человека подробно.'

# ========== ANTI-PREFERENCES PROMPT ==========
ANTI_SYSTEM = """Ты -- психолог-аналитик. Определи, что НЕЛЬЗЯ дарить этому человеку.
Проанализируй портрет и выдели категории подарков, которые категорически не подходят.
Верни список из 3-5 пунктов с краткими пояснениями."""

def anti_prompt(query, persona):
    return f"""👤 ПОРТРЕТ ПОЛУЧАТЕЛЯ:
{persona}

Исходный запрос: "{query}"

Проанализируй портрет и определи:
1. Что человек точно НЕ любит (из запроса)
2. Что не соответствует его характеру
3. Что может вызвать негативные эмоции

Верни список "анти-подарков" с пояснениями:"""

# ========== CREATIVE GENERATOR ==========
CREATIVE_SYSTEM = """Ты -- креативный эксперт по подаркам с 20-летним опытом. 
На основе портрета человека и списка того, что НЕЛЬЗЯ дарить, придумай 5 ОРИГИНАЛЬНЫХ идей подарков.

ПРАВИЛА:
1. НЕ копируй названия из списка товаров - создавай новые идеи
2. Каждая идея должна быть персонализированной под конкретного человека
3. Учитывай бюджет, если он указан
4. Строго избегай того, что человек не любит
5. Идеи должны быть реалистичными и доступными на российских маркетплейсах
6. Названия подарков должны быть ТЕХНИЧЕСКИМИ и КОНКРЕТНЫМИ на русском языке (например, "Беспроводные наушники", "Механическая клавиатура", "Кухонный комбайн")

Формат каждой идеи:
ПОДАРОК: [название подарка]
ПОЧЕМУ: [почему этот подарок идеально подходит этому человеку]
МОМЕНТ: [как будет использоваться, какие эмоции вызовет]
ЧЕМ НЕ БАНАЛЬНО: [почему это особенный подарок]

ВАЖНО: Идеи должны быть РАЗНЫМИ и УНИКАЛЬНЫМИ!"""

def creative_prompt(query, persona, anti_list, products, budget=None, occasion=None):
    products_list = "\n".join([f"  • {p}" for p in products[:10]])
    anti_list_text = "\n".join([f"  • {item}" for item in anti_list[:5]])
    
    ctx = ""
    if occasion:
        ctx += f"Повод: {occasion}\n"
    if budget:
        ctx += f"Бюджет: {budget}\n"
    
    return f"""👤 ПОРТРЕТ ПОЛУЧАТЕЛЯ:
{persona}

🚫 ЧТО НЕЛЬЗЯ ДАРИТЬ:
{anti_list_text}

{ctx}📦 ТОВАРЫ ДЛЯ ВДОХНОВЕНИЯ (используй как референсы, но не копируй):
{products_list}

🎨 Придумай 5 ОРИГИНАЛЬНЫХ идей подарков для этого человека.
Используй товары только как источник вдохновения для понимания доступных категорий.

Требования к идеям:
- Персонализированные (связаны с характером человека)
- Оригинальные (не банальные носки/кружки/сувениры)
- Учитывают скрытые желания из портрета
- Практичные или эмоциональные
- Реалистичные для покупки
- Названия ТОЛЬКО на русском, технические и конкретные

Начинай сразу с идей:"""

# ========== ПАРСИНГ ИДЕЙ ==========
def parse_ideas(creative_response):
    ideas = []
    current_idea = []
    
    for line in creative_response.split('\n'):
        if line.startswith('ПОДАРОК:'):
            if current_idea:
                ideas.append('\n'.join(current_idea))
                current_idea = []
            current_idea.append(line)
        elif line.startswith(('ПОЧЕМУ:', 'МОМЕНТ:', 'ЧЕМ НЕ БАНАЛЬНО:')):
            if line.strip():
                current_idea.append(line)
        elif line.strip() and current_idea and not line.startswith('---'):
            current_idea.append(line)
    
    if current_idea:
        ideas.append('\n'.join(current_idea))
    
    return ideas[:5]

def extract_idea_title(idea_text):
    """Извлекает название подарка из текста идеи"""
    for line in idea_text.split('\n'):
        if line.startswith('ПОДАРОК:'):
            return line.replace('ПОДАРОК:', '').strip().strip('*').strip()
    return ""

def extract_ideas_to_df(ideas):
    """Извлекает названия и описания в DataFrame для получения цен"""
    gifts = []
    for idea in ideas:
        lines = idea.split('\n')
        title = ""
        description = ""
        for line in lines:
            if line.startswith('ПОДАРОК:'):
                title = line.replace('ПОДАРОК:', '').strip().strip('*').strip()
            elif line.startswith('ПОЧЕМУ:'):
                description = line.replace('ПОЧЕМУ:', '').strip()
        if title:
            gifts.append({"товар": title, "описание": description})
    return pd.DataFrame(gifts)

def create_idea_with_price(idea_text, price):
    """Добавляет цену к тексту идеи"""
    lines = idea_text.split('\n')
    # Вставляем цену после названия
    for i, line in enumerate(lines):
        if line.startswith('ПОДАРОК:'):
            lines.insert(i + 1, f"💰 **Цена:** {price}")
            break
    return '\n'.join(lines)

# ========== MAIN FUNCTION ==========
def gift_agent(query, verbose=True, get_prices=True):
    if verbose:
        print(f"\n🔍 Обработка запроса: {query}")
    
    budget = extract_budget(query)
    occasion = detect_occasion(query)
    
    if is_query_too_vague(query):
        return {"status": "needs_clarification", "questions": "Уточните, пожалуйста, кому подарок и какие интересы?"}
    
    if verbose: print("[1/5] Поиск товаров в базе...")
    idx = retrieve_for_query(query)
    products, context_df = get_products_context(idx)
    
    if verbose: print(f"   Найдено {len(products)} товаров для вдохновения")
    
    if verbose: print("[2/5] Создание портрета через Groq...")
    persona = call_groq(PERSONA_SYSTEM, persona_prompt(query, occasion, budget), 500)
    
    if not persona:
        persona = "Человек, который ищет подарок. Подробности неизвестны."
    
    if verbose: print("[3/5] Анализ того, что НЕЛЬЗЯ дарить...")
    anti_response = call_groq(ANTI_SYSTEM, anti_prompt(query, persona), 400)
    anti_list = [line.strip() for line in anti_response.split('\n') if line.strip() and not line.startswith('---')]
    
    if not anti_list:
        anti_list = ["Избегайте банальных подарков (носки, кружки, сувениры)"]
    
    if verbose: print("[4/5] Генерация оригинальных идей через Groq...")
    creative_response = call_groq(
        CREATIVE_SYSTEM,
        creative_prompt(query, persona, anti_list, products, budget, occasion),
        1200
    )
    
    if not creative_response:
        if verbose: print("⚠️ Groq не вернул ответ, используем запасной вариант")
        return {
            "status": "fallback",
            "ideas": [f"ПОДАРОК: {p}\nПОЧЕМУ: Подходит по интересам\nМОМЕНТ: Будет рад получить\nЧЕМ НЕ БАНАЛЬНО: Персонализированный подарок" for p in products[:5]],
            "budget": budget,
            "occasion": occasion,
            "persona": persona
        }
    
    ideas = parse_ideas(creative_response)
    
    # Получаем цены, если нужно
    if get_prices:
        if verbose: print("[5/5] Получение цен на Яндекс Маркете...")
        df_gifts = extract_ideas_to_df(ideas)
        
        prices = []
        for idx, row in df_gifts.iterrows():
            product = row['товар']
            if verbose: print(f"   → {product[:50]}...")
            price = get_price(product, "yandex")
            prices.append(price if price else "Цена не найдена")
            time.sleep(random.uniform(2, 3))  # Задержка между запросами
        
        df_gifts['цена'] = prices
        
        # Добавляем цены к идеям
        final_ideas = []
        for i, idea in enumerate(ideas):
            if i < len(df_gifts):
                final_ideas.append(create_idea_with_price(idea, df_gifts.iloc[i]['цена']))
            else:
                final_ideas.append(idea)
    else:
        # Просто добавляем ссылки без цен
        final_ideas = []
        for idea in ideas:
            idea_title = extract_idea_title(idea)
            links_markdown = create_search_links_for_idea(idea_title)
            final_ideas.append(idea + links_markdown)
    
    if verbose:
        print(f"[5/5] Сгенерировано {len(final_ideas)} идей")
        if get_prices:
            print("   💰 Цены получены для", len([p for p in prices if p]), "товаров")
    
    return {
        "status": "success",
        "ideas": final_ideas,
        "budget": budget,
        "occasion": occasion,
        "persona": persona
    }

def create_search_links_for_idea(idea_title: str) -> str:
    """Создает Markdown-строку со ссылками на поиск идеи на маркетплейсах"""
    clean_title = idea_title.strip().strip('*').strip('"')
    
    wb_link = f"https://www.wildberries.ru/catalog/0/search.aspx?search={urllib.parse.quote(clean_title)}"
    ym_link = f"https://market.yandex.ru/search?text={urllib.parse.quote(clean_title).replace('%20', '+')}"
    ozon_link = f"https://www.ozon.ru/search/?text={urllib.parse.quote(clean_title)}"
    
    return (f"\n\n🔗 **Найти этот подарок:**\n"
            f"  • [Wildberries]({wb_link}) | "
            f"[Яндекс Маркет]({ym_link}) | "
            f"[Ozon]({ozon_link})")

def get_product_links(product_name):
    """Генерирует ссылки на конкретный товар (для обратной совместимости)"""
    return create_all_links(product_name)
    
# ========== API FUNCTION ==========
def generate_gifts(description):
    try:
        # Параметр get_prices=True включает получение цен
        result = gift_agent(description, verbose=False, get_prices=True)
        if result["status"] == "needs_clarification":
            return [f"Уточните: {result['questions']}"]
        return result.get("ideas", ["Идеи не найдены"])
    except Exception as e:
        print(f"❌ Ошибка: {e}")
        return [f"Ошибка: {e}"]

print("\n" + "=" * 60)
print("✅ GIFT GENIUS С ГЕНЕРАЦИЕЙ ИДЕЙ И ЦЕНАМИ ГОТОВ!")
print("=" * 60)
print("🎯 Теперь для каждой идеи генерируются ссылки и цены")
print("=" * 60)

Overwriting rag_pipeline.py


In [7]:
%%writefile api.py

from fastapi import FastAPI
from pydantic import BaseModel
from rag_pipeline import generate_gifts
import logging
import time

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI(title="GiftGenius API")

class Request(BaseModel):
    description: str

class Response(BaseModel):
    ideas: list
    processing_time: float = None
    status: str = "success"

@app.get("/")
def root():
    return {"message": "🎁 Gift Genius v5 + Groq", "status": "healthy"}

@app.get("/health")
def health():
    return {"status": "healthy", "timestamp": time.time()}

@app.post("/generate", response_model=Response)
def generate(request: Request):
    start = time.time()
    logger.info(f"📨 Запрос: {request.description}")
    
    try:
        ideas = generate_gifts(request.description)
        return {
            "ideas": ideas,
            "processing_time": time.time() - start,
            "status": "success"
        }
    except Exception as e:
        logger.error(f"❌ Ошибка: {e}")
        return {
            "ideas": [f"Ошибка: {e}"],
            "processing_time": time.time() - start,
            "status": "error"
        }

Writing api.py


In [23]:
import uvicorn
from threading import Thread
import time

def run_server():
    uvicorn.run("api:app", host="0.0.0.0", port=8000, log_level="info")

# Останавливаем старые процессы
import os
os.system("pkill -f uvicorn")
os.system("pkill -f ngrok")
time.sleep(2)

# Запускаем сервер
server_thread = Thread(target=run_server)
server_thread.daemon = True
server_thread.start()
time.sleep(3)
print("✅ Сервер запущен")

INFO:     Started server process [55]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ Сервер запущен


In [24]:
from pyngrok import ngrok


ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()

public_url = ngrok.connect(8000)
print("="*60)
print("🎁 GIFT GENIUS v5 + GROQ API ГОТОВ!")
print("="*60)
print(f"🌐 URL: {public_url}")
print(f"📌 POST {public_url}/generate")
print("="*60)

INFO:pyngrok.process:Updating authtoken for default "config_path" of "ngrok_path": /root/.config/ngrok/ngrok
INFO:pyngrok.ngrok:Opening tunnel named: http-8000-ef3ca9eb-ad6f-4d23-81a0-e6ecaefdaf52
INFO:pyngrok.process.ngrok:t=2026-03-13T15:08:36+0000 lvl=info msg="no configuration paths supplied"
INFO:pyngrok.process.ngrok:t=2026-03-13T15:08:36+0000 lvl=info msg="using configuration at default config path" path=/root/.config/ngrok/ngrok.yml
INFO:pyngrok.process.ngrok:t=2026-03-13T15:08:36+0000 lvl=info msg="open config file" path=/root/.config/ngrok/ngrok.yml err=nil
INFO:pyngrok.process.ngrok:t=2026-03-13T15:08:36+0000 lvl=info msg="FIPS 140 mode" enabled=false
INFO:pyngrok.process.ngrok:t=2026-03-13T15:08:36+0000 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
INFO:pyngrok.process.ngrok:t=2026-03-13T15:08:36+0000 lvl=info msg="client session established" obj=tunnels.session
INFO:pyngrok.process.ngrok:t=2026-03-13T15:08:36+0000 lvl=info msg="tunnel sessi

🎁 GIFT GENIUS v5 + GROQ API ГОТОВ!
🌐 URL: NgrokTunnel: "https://walker-unerasable-will.ngrok-free.dev" -> "http://localhost:8000"
📌 POST NgrokTunnel: "https://walker-unerasable-will.ngrok-free.dev" -> "http://localhost:8000"/generate


INFO:pyngrok.process.ngrok:t=2026-03-13T15:08:36+0000 lvl=info msg=end pg=/api/tunnels id=63aa148b071ac273 status=201 dur=290.556595ms


In [18]:
import rag_pipeline


test_query = "Маме 50 лет, любит готовить и читать. Бюджет до 3000 рублей. Не любит сладкое и украшения."

result = rag_pipeline.gift_agent(test_query, verbose=True)

print("\n" + "="*60)
print("📊 ИТОГОВЫЙ РЕЗУЛЬТАТ")
print("="*60)
for i, idea in enumerate(result['ideas'], 1):
    print(f"\n--- ИДЕЯ {i} ---")
    print(idea)

Устройство: cuda

📊 ЗАГРУЗКА ДАННЫХ


✅ Подарков в базе: 305

🔤 ЗАГРУЗКА EMBEDDING МОДЕЛИ


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

✅ Модель загружена

🔍 СОЗДАНИЕ ИНДЕКСОВ
✅ Эмбеддинги: (305, 1024)
✅ Индексы готовы

✅ GIFT GENIUS С ГЕНЕРАЦИЕЙ ИДЕЙ И ЦЕНАМИ ГОТОВ!
🎯 Теперь для каждой идеи генерируются ссылки и цены

🔍 Обработка запроса: Маме 50 лет, любит готовить и читать. Бюджет до 3000 рублей. Не любит сладкое и украшения.
[1/5] Поиск товаров в базе...
   Найдено 15 товаров для вдохновения
[2/5] Создание портрета через Groq...
[3/5] Анализ того, что НЕЛЬЗЯ дарить...
[4/5] Генерация оригинальных идей через Groq...
[5/5] Получение цен на Яндекс Маркете...
   → Автоматическая линия для приготовления суши...
   → Электронная книга с сенсорным экраном...
   → Набор ручных форм для выпечки хлеба...
   → Портативная мини- печь для кулинарии...
   → Комплект для создания домашних консервов...
[5/5] Сгенерировано 5 идей
   💰 Цены получены для 5 товаров

📊 ИТОГОВЫЙ РЕЗУЛЬТАТ

--- ИДЕЯ 1 ---
ПОДАРОК: Автоматическая линия для приготовления суши
💰 **Цена:** Цена не найдена
ПОЧЕМУ: Этот подарок идеально подходит для женщины, у

t=2026-03-13T15:00:04+0000 lvl=warn msg="failed to open private leg" id=8086ea5ef613 privaddr=localhost:8000 err="dial tcp [::1]:8000: connect: connection refused"


In [25]:
from pyngrok import ngrok

# Посмотреть все активные туннели
tunnels = ngrok.get_tunnels()
for tunnel in tunnels:
    print(f"URL: {tunnel.public_url}")
    print(f"-> {tunnel.config['addr']}")

INFO:pyngrok.process.ngrok:t=2026-03-13T15:08:52+0000 lvl=info msg=start pg=/api/tunnels id=a1380f37c9ef76f9


URL: https://walker-unerasable-will.ngrok-free.dev
-> http://localhost:8000


INFO:pyngrok.process.ngrok:t=2026-03-13T15:08:52+0000 lvl=info msg=end pg=/api/tunnels id=a1380f37c9ef76f9 status=200 dur=205.725µs


In [27]:
import requests

# Проверка локального сервера
try:
    response = requests.get("http://localhost:8000/health", timeout=5)
    print(f"Локальный статус: {response.status_code}")
    print(f"Ответ: {response.json()}")
except Exception as e:
    print(f"❌ Локальный сервер не отвечает: {e}")

INFO:     127.0.0.1:51996 - "GET /health HTTP/1.1" 200 OK
Локальный статус: 200
Ответ: {'status': 'healthy', 'timestamp': 1773414543.6069715}


INFO:pyngrok.process.ngrok:t=2026-03-13T15:09:21+0000 lvl=info msg="join connections" obj=join id=5c114d6770da l=127.0.0.1:8000 r=121.127.46.69:34368
INFO:api:📨 Запрос: Папа, 55 лет, рыбалка, дача, грибы


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


INFO:     121.127.46.69:0 - "POST /generate HTTP/1.1" 200 OK


...

INFO:     121.127.46.69:0 - "POST /generate HTTP/1.1" 200 OK


INFO:pyngrok.process.ngrok:t=2026-03-13T15:41:45+0000 lvl=info msg="join connections" obj=join id=bac4797584f5 l=127.0.0.1:8000 r=121.127.46.69:7834
INFO:api:📨 Запрос: Коллега, 40 лет, гитарист, любит готовить


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


INFO:     121.127.46.69:0 - "POST /generate HTTP/1.1" 200 OK


In [ ]:
# # Проверьте, какие эндпоинты доступны
# import requests

# base_url = "https://walker-unerasable-will.ngrok-free.dev"

# # Попробуйте разные варианты
# endpoints = [
#     "/health",
#     "/",
#     "/docs",
#     "/openapi.json"
# ]

# for endpoint in endpoints:
#     try:
#         r = requests.get(f"{base_url}{endpoint}", timeout=3)
#         print(f"{endpoint}: {r.status_code}")
#     except:
#         print(f"{endpoint}: ошибка")

In [22]:
import requests
import subprocess
import time

print("="*50)
print("🔍 ДИАГНОСТИКА СЕРВЕРА")
print("="*50)

# 1. Проверка запущен ли Uvicorn процесс
print("\n📊 Проверка процессов:")
result = subprocess.run(['ps', 'aux'], capture_output=True, text=True)
if 'uvicorn' in result.stdout:
    print("✅ Uvicorn процесс найден")
    # Покажем детали процесса
    for line in result.stdout.split('\n'):
        if 'uvicorn' in line:
            print(f"   {line[:100]}")
else:
    print("❌ Uvicorn процесс НЕ найден - сервер не запущен!")

# 2. Проверка локального сервера
print("\n🌐 Проверка локального сервера (http://localhost:8000):")
try:
    response = requests.get("http://localhost:8000/health", timeout=5)
    print(f"✅ Локальный сервер отвечает: {response.status_code}")
    print(f"   Ответ: {response.json()}")
except requests.exceptions.ConnectionError:
    print("❌ Локальный сервер НЕ доступен! Port 8000 не отвечает")
except Exception as e:
    print(f"❌ Ошибка: {e}")

# 3. Проверка ngrok туннеля
print("\n🔄 Проверка ngrok туннеля:")
try:
    response = requests.get("https://walker-unerasable-will.ngrok-free.dev/health", timeout=10)
    print(f"✅ Ngrok туннель работает: {response.status_code}")
    print(f"   Ответ: {response.json()}")
except requests.exceptions.ConnectionError:
    print("❌ Ngrok туннель НЕ работает! Возможно истекло время")
except Exception as e:
    print(f"❌ Ошибка: {e}")

# 4. Проверка доступных эндпоинтов
print("\n📌 Проверка эндпоинтов:")
endpoints = ["/", "/health", "/docs", "/openapi.json", "/generate"]
base_url = "https://walker-unerasable-will.ngrok-free.dev"

for endpoint in endpoints:
    try:
        r = requests.get(f"{base_url}{endpoint}", timeout=5)
        print(f"{endpoint:15} -> {r.status_code}")
    except:
        print(f"{endpoint:15} -> ❌ ошибка")

print("\n="*50)

t=2026-03-13T15:07:29+0000 lvl=warn msg="failed to open private leg" id=370cd094957c privaddr=localhost:8000 err="dial tcp [::1]:8000: connect: connection refused"


🔍 ДИАГНОСТИКА СЕРВЕРА

📊 Проверка процессов:
❌ Uvicorn процесс НЕ найден - сервер не запущен!

🌐 Проверка локального сервера (http://localhost:8000):
❌ Локальный сервер НЕ доступен! Port 8000 не отвечает

🔄 Проверка ngrok туннеля:
✅ Ngrok туннель работает: 502
❌ Ошибка: Expecting value: line 1 column 1 (char 0)

📌 Проверка эндпоинтов:


t=2026-03-13T15:07:29+0000 lvl=warn msg="failed to open private leg" id=c600b06f5210 privaddr=localhost:8000 err="dial tcp [::1]:8000: connect: connection refused"
t=2026-03-13T15:07:29+0000 lvl=warn msg="failed to open private leg" id=840423ba119e privaddr=localhost:8000 err="dial tcp [::1]:8000: connect: connection refused"
t=2026-03-13T15:07:29+0000 lvl=warn msg="failed to open private leg" id=9144b4be96cf privaddr=localhost:8000 err="dial tcp [::1]:8000: connect: connection refused"


/               -> 502
/health         -> 502
/docs           -> 502


t=2026-03-13T15:07:29+0000 lvl=warn msg="failed to open private leg" id=84ae3bf4d96c privaddr=localhost:8000 err="dial tcp [::1]:8000: connect: connection refused"
t=2026-03-13T15:07:29+0000 lvl=warn msg="failed to open private leg" id=21ad3005dcd8 privaddr=localhost:8000 err="dial tcp [::1]:8000: connect: connection refused"


/openapi.json   -> 502
/generate       -> 502

=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
